In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import current_timestamp, col

In [0]:
telemetry_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("speed", DoubleType(), True),
    StructField("engine_temperature", DoubleType(), True),
    StructField("fuel_level", DoubleType(), True),
    StructField("battery_voltage", DoubleType(), True),
    StructField("event_type", StringType(), True),
    StructField("sensor_status", StringType(), True)
])

In [0]:
raw_path = "/Volumes/dev/iot_fleet/raw/telemetry"
bronze_checkpoint = "/Volumes/dev/iot_fleet/raw/checkpoints/bronze"


In [0]:
bronze_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .schema(telemetry_schema) \
    .load(raw_path) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("ingestion_timestamp", current_timestamp())


In [0]:
bronze_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", bronze_checkpoint) \
    .outputMode("append") \
    .trigger(availableNow = True) \
    .toTable("dev.iot_fleet.bronze_telemetry")

In [0]:
display(spark.table("dev.iot_fleet.bronze_telemetry").limit(5))

In [0]:
spark.sql("select count(*) from dev.iot_fleet.bronze_telemetry").show()

In [0]:
#source file tracking
spark.sql("""SELECT source_file, COUNT(*) AS records FROM dev.iot_fleet.bronze_telemetry GROUP BY source_file""").show(truncate=False)

In [0]:
spark.table("dev.iot_fleet.bronze_telemetry").printSchema()